# Introduction

This notebook is for EDA of the AIHW MyHospitals API data. The link to the documentation can be found here:
https://www.aihw.gov.au/hospitals/other-resources/myhospitals-api

The base URL:
https://myhospitalsapi.aihw.gov.au/ 

The notebook will be structured in the following format:
- Obtaining measures
- Pulling data from identified relevent measures for the project
- Rudimentary Cleaning w/ Pandas
- Data exploration w/ tables, summaries, distribution charts etc.

# 1. Inspect Available Measures

The API hierarchy is as follows:
- Measures-Categories
    - Measures (A calculated statistic)
        - Reported-Measures (Disaggregated Statistics, more granular, can query)
- Reporting-Units (Used to identify the meaning of data values)
- datasets (Gives csv in zipped format)

To use the latest API, we use the "api/v1/..." endpoint along any of the above to fetch the data.



In [1]:
import json
from urllib.request import Request, urlopen

MEASURE_CATEGORIES_URL = "https://myhospitalsapi.aihw.gov.au/api/v1/measure-categories"

request = Request(
    MEASURE_CATEGORIES_URL,
    headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
)

with urlopen(request, timeout=30) as response:
    measure_categories = json.load(response)

print(json.dumps(measure_categories, indent=2))

{
  "result": [
    {
      "measure_category_code": "MYH-ADM",
      "measure_category_name": "MyHospitals Admissions"
    },
    {
      "measure_category_code": "MYH-CANCER",
      "measure_category_name": "MyHospitals Cancer"
    },
    {
      "measure_category_code": "MYH-CWS",
      "measure_category_name": "MyHospitals Financial performance"
    },
    {
      "measure_category_code": "MYH-ED",
      "measure_category_name": "MyHospitals ED Presentations"
    },
    {
      "measure_category_code": "MYH-ED-TIME",
      "measure_category_name": "MyHospitals Time in ED"
    },
    {
      "measure_category_code": "MYH-ED-WAITS",
      "measure_category_name": "MyHospitals ED Waiting Times"
    },
    {
      "measure_category_code": "MYH-ES",
      "measure_category_name": "MyHospitals Elective surgery"
    },
    {
      "measure_category_code": "MYH-HH",
      "measure_category_name": "MyHospitals Hand hygiene"
    },
    {
      "measure_category_code": "MYH-LOS",
      "measu

### Primary Measures for Emergency Department
- ED Presentations
- Time in ED
- ED Waiting Times

### Secondary Measures for Emergency Department
- Length of Stay
- Relative Stay Index
- Hospital Admissions
- Specialised Services
- Elective Surgery

### Low Priority
- Financial Performance
- MyHospitals SAB (Staph infections)

In [4]:
from html import escape
from IPython.display import HTML, display

MEASURES_URL = (
    "https://myhospitalsapi.aihw.gov.au/api/v1/measure-categories"
    "/{category_code}/measures"
)
ED_MEASURE_CATEGORIES = {
    "MYH-ED": "ED Presentations",
    "MYH-ED-TIME": "Time in ED",
    "MYH-ED-WAITS": "ED Waiting Times",
}
TIME_GRANULARITY = "Annual (financial year: 1 July–30 June)"

measures_by_category = {}
ed_measure_codes = []
for category_code, category_name in ED_MEASURE_CATEGORIES.items():
    request = Request(
        MEASURES_URL.format(category_code=category_code),
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
    )
    with urlopen(request, timeout=30) as response:
        measures = json.load(response)["result"]

    ed_measure_codes.extend(measure["measure_code"] for measure in measures)
    measures_by_category[category_name] = [
        (
            f"{measure['measure_code']} — {measure['measure_name']}"
            f" | Resolution: {TIME_GRANULARITY}"
        )
        for measure in measures
    ]

row_count = max(map(len, measures_by_category.values()))
header_html = "".join(
    f"<th style='text-align:left'>{escape(category)}</th>"
    for category in measures_by_category
)
rows_html = "".join(
    "<tr>"
    + "".join(
        f"<td style='text-align:left;vertical-align:top'>{escape(measures[row]) if row < len(measures) else ''}</td>"
        for measures in measures_by_category.values()
    )
    + "</tr>"
    for row in range(row_count)
)

display(HTML(f"<table><thead><tr>{header_html}</tr></thead><tbody>{rows_html}</tbody></table>"))

ED Presentations,Time in ED,ED Waiting Times
MYH0035 — Number of patients presenting to the emergency department | Resolution: Annual (financial year: 1 July–30 June),MYH0005 — Percentage of patients who depart the emergency department within four hours of arrival | Resolution: Annual (financial year: 1 July–30 June),MYH0010 — Percentage of patients who commenced treatment within the recommended time | Resolution: Annual (financial year: 1 July–30 June)
,MYH0012 — Number of patients presenting to the emergency department | Resolution: Annual (financial year: 1 July–30 June),MYH0011 — Number of patients presenting to the emergency department | Resolution: Annual (financial year: 1 July–30 June)
,MYH0013 — Time until most patients (90%) departed the emergency department | Resolution: Annual (financial year: 1 July–30 June),
,MYH0036 — Median time (50%) patients departed emergency department | Resolution: Annual (financial year: 1 July–30 June),


## Datasets

In [14]:
import json
from html import escape
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from IPython.display import HTML, display

DATASETS_URL = "https://myhospitalsapi.aihw.gov.au/api/v1/datasets"
ed_measure_codes = (
    "MYH0035",  # ED presentations
    "MYH0005", "MYH0012", "MYH0013", "MYH0036",  # Time in ED
    "MYH0010", "MYH0011",  # ED waiting times
)
datasets = []
for measure_code in ed_measure_codes:
    request = Request(
        f"{DATASETS_URL}?{urlencode({'measure_code': measure_code})}",
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
    )
    with urlopen(request, timeout=30) as response:
        datasets.extend(json.load(response)["result"])

datasets = sorted(
    datasets,
    key=lambda dataset: (dataset["reporting_end_date"], dataset["data_set_id"]),
    reverse=True,
)

dataset_rows_html = "".join(
    "<tr>"
    f"<td>{dataset['data_set_id']}</td>"
    f"<td>{escape(dataset['reporting_start_date'])}</td>"
    f"<td>{escape(dataset['reporting_end_date'])}</td>"
    f"<td>{escape(dataset['reported_measure_summary']['measure_summary']['measure_code'])}</td>"
    f"<td>{escape(dataset['reported_measure_summary']['measure_summary']['measure_name'])}</td>"
    f"<td>{escape(dataset['reported_measure_summary']['reported_measure_code'])}</td>"
    f"<td>{escape(dataset['reported_measure_summary']['reported_measure_name'])}</td>"
    f"<td>{escape(dataset['data_set_name'])}</td>"
    "</tr>"
    for dataset in datasets
)

display(HTML(
    f"<p><strong>{len(datasets):,}</strong> available ED datasets (newest first)</p>"
    "<div style='max-height:600px;overflow:auto'>"
    "<table><thead><tr>"
    "<th>Dataset ID</th><th>Start</th><th>End</th><th>Measure code</th>"
    "<th>Measure</th><th>Reported-measure code</th>"
    "<th>Reported measure</th><th>Dataset name</th>"
    f"</tr></thead><tbody>{dataset_rows_html}</tbody></table></div>"
))

Dataset ID,Start,End,Measure code,Measure,Reported-measure code,Reported measure,Dataset name
11192,2024-07-01,2025-06-30,MYH0013,Time until most patients (90%) departed the emergency department,MYH-RM0302,All patients,ED-TIME 2024-25 - Time until most patients (90%) departed the emergency department
11191,2024-07-01,2025-06-30,MYH0013,Time until most patients (90%) departed the emergency department,MYH-RM0301,Not subsequently admitted patients,ED-TIME 2024-25 - Time until most patients (90%) departed the emergency department
11190,2024-07-01,2025-06-30,MYH0036,Median time (50%) patients departed emergency department,MYH-RM0300,All patients,ED-TIME 2024-25 - Median time (50%) patients departed emergency department
11189,2024-07-01,2025-06-30,MYH0036,Median time (50%) patients departed emergency department,MYH-RM0299,Not subsequently admitted patients,ED-TIME 2024-25 - Median time (50%) patients departed emergency department
11188,2024-07-01,2025-06-30,MYH0036,Median time (50%) patients departed emergency department,MYH-RM0298,Subsequently admitted patients,ED-TIME 2024-25 - Median time (50%) patients departed emergency department
11187,2024-07-01,2025-06-30,MYH0035,Number of patients presenting to the emergency department,MYH-RM0297,Non-Urgent,ED 2024-25 - Number of patients presenting to the emergency department
11186,2024-07-01,2025-06-30,MYH0035,Number of patients presenting to the emergency department,MYH-RM0296,Semi-Urgent,ED 2024-25 - Number of patients presenting to the emergency department
11185,2024-07-01,2025-06-30,MYH0035,Number of patients presenting to the emergency department,MYH-RM0295,Urgent,ED 2024-25 - Number of patients presenting to the emergency department
11184,2024-07-01,2025-06-30,MYH0035,Number of patients presenting to the emergency department,MYH-RM0294,Emergency,ED 2024-25 - Number of patients presenting to the emergency department
11183,2024-07-01,2025-06-30,MYH0035,Number of patients presenting to the emergency department,MYH-RM0293,Resuscitation,ED 2024-25 - Number of patients presenting to the emergency department


In [15]:
from collections import defaultdict
from datetime import date
from html import escape

from IPython.display import HTML, display

if "datasets" not in globals():
    raise RuntimeError("Run the preceding ED datasets cell first.")

datasets_by_measure = defaultdict(list)
for dataset in datasets:
    measure = dataset["reported_measure_summary"]["measure_summary"]
    datasets_by_measure[(measure["measure_code"], measure["measure_name"])].append(dataset)

def describe_granularity(measure_datasets):
    period_lengths = {
        (date.fromisoformat(item["reporting_end_date"])
         - date.fromisoformat(item["reporting_start_date"])).days + 1
        for item in measure_datasets
    }
    is_financial_year = all(
        item["reporting_start_date"][5:] == "07-01"
        and item["reporting_end_date"][5:] == "06-30"
        for item in measure_datasets
    )
    if period_lengths <= {365, 366} and is_financial_year:
        return "Annual (financial year)"
    return "Mixed reporting periods"

summary_rows = []
for (measure_code, measure_name), measure_datasets in sorted(datasets_by_measure.items()):
    earliest_start = min(item["reporting_start_date"] for item in measure_datasets)
    latest_end = max(item["reporting_end_date"] for item in measure_datasets)
    first_year = int(earliest_start[:4])
    last_year = int(latest_end[:4]) - 1
    year_range = (
        f"{first_year}–{str(first_year + 1)[-2:]} "
        f"to {last_year}–{str(last_year + 1)[-2:]}"
    )
    summary_rows.append(
        (measure_code, measure_name, len(measure_datasets), year_range, describe_granularity(measure_datasets))
    )

summary_rows_html = "".join(
    "<tr>"
    f"<td>{escape(measure_code)}</td>"
    f"<td>{escape(measure_name)}</td>"
    f"<td>{dataset_count:,}</td>"
    f"<td>{escape(year_range)}</td>"
    f"<td>{escape(granularity)}</td>"
    "</tr>"
    for measure_code, measure_name, dataset_count, year_range, granularity in summary_rows
)

display(HTML(
    "<table><thead><tr>"
    "<th>Measure code</th><th>Measure</th><th>Number of datasets</th>"
    "<th>Available year range</th><th>Time granularity</th>"
    f"</tr></thead><tbody>{summary_rows_html}</tbody></table>"
))

Measure code,Measure,Number of datasets,Available year range,Time granularity
MYH0005,Percentage of patients who depart the emergency department within four hours of arrival,112,2011–12 to 2024–25,Annual (financial year)
MYH0010,Percentage of patients who commenced treatment within the recommended time,70,2011–12 to 2024–25,Annual (financial year)
MYH0011,Number of patients presenting to the emergency department,70,2011–12 to 2024–25,Annual (financial year)
MYH0012,Number of patients presenting to the emergency department,112,2011–12 to 2024–25,Annual (financial year)
MYH0013,Time until most patients (90%) departed the emergency department,42,2011–12 to 2024–25,Annual (financial year)
MYH0035,Number of patients presenting to the emergency department,70,2011–12 to 2024–25,Annual (financial year)
MYH0036,Median time (50%) patients departed emergency department,42,2011–12 to 2024–25,Annual (financial year)


In [17]:
import json
from datetime import date
from html import escape
from urllib.request import Request, urlopen

from IPython.display import HTML, display

DATASET_ID = 11192
DATASET_URL = f"https://myhospitalsapi.aihw.gov.au/api/v1/datasets/{DATASET_ID}"
REQUEST_HEADERS = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}

def fetch_result(url):
    request = Request(url, headers=REQUEST_HEADERS)
    with urlopen(request, timeout=30) as response:
        return json.load(response)["result"]

dataset_11192 = fetch_result(DATASET_URL)
dataset_11192_items = fetch_result(f"{DATASET_URL}/data-items")

start_date = date.fromisoformat(dataset_11192["reporting_start_date"])
end_date = date.fromisoformat(dataset_11192["reporting_end_date"])
period_days = (end_date - start_date).days + 1
granularity = (
    "Annual (financial year)"
    if start_date.month == 7 and start_date.day == 1
    and end_date.month == 6 and end_date.day == 30
    and period_days in {365, 366}
    else "Other"
)

data_item_rows_html = "".join(
    "<tr>"
    f"<td>{escape(str((item.get('reporting_unit_summary') or {}).get('reporting_unit_code', '')))}</td>"
    f"<td>{escape(str((item.get('reporting_unit_summary') or {}).get('reporting_unit_name', '')))}</td>"
    f"<td>{escape(str(((item.get('reporting_unit_summary') or {}).get('reporting_unit_type') or {}).get('reporting_unit_type_name', '')))}</td>"
    f"<td>{escape(str((item.get('peer_group_summary') or {}).get('reporting_unit_name', '')))}</td>"
    f"<td>{escape(str(item.get('value', '')))}</td>"
    "</tr>"
    for item in dataset_11192_items
)

display(HTML(
    f"<h4>Dataset {DATASET_ID}</h4>"
    f"<p><strong>{escape(dataset_11192['data_set_name'])}</strong></p>"
    f"<p>Reported measure: <strong>{escape(dataset_11192['reported_measure_summary']['reported_measure_code'])} — "
    f"{escape(dataset_11192['reported_measure_summary']['reported_measure_name'])}</strong></p>"
    f"<p>Reporting period: {start_date} to {end_date} ({period_days} days)<br>"
    f"Time granularity: <strong>{granularity}</strong><br>"
    f"Data points: {len(dataset_11192_items):,}</p>"
    "<div style='max-height:600px;overflow:auto'>"
    "<table><thead><tr><th>Reporting-unit code</th><th>Reporting unit</th>"
    "<th>Unit type</th><th>Peer group</th><th>Value</th>"
    f"</tr></thead><tbody>{data_item_rows_html}</tbody></table></div>"
))

Reporting-unit code,Reporting unit,Unit type,Peer group,Value
H0014,The Children's Hospital at Westmead,Hospital,Children's hospitals,475.0
H0015,Sydney Children's Hospital,Hospital,Children's hospitals,404.0
H0017,St Vincent's Hospital [Darlinghurst],Hospital,Major hospitals,935.0
H0020,Canterbury Hospital,Hospital,Medium metropolitan hospitals,463.0
H0021,Royal Prince Alfred Hospital,Hospital,Major hospitals,787.0
H0023,Concord Repatriation Hospital,Hospital,Major hospitals,600.0
H0026,Camden Hospital,Hospital,Unpeered,300.0
H0027,Fairfield Hospital,Hospital,Medium metropolitan hospitals,674.0
H0028,Liverpool Hospital,Hospital,Major hospitals,902.0
H0029,Campbelltown Hospital,Hospital,Large metropolitan hospitals,933.0
